In [24]:
import pandas as pd
import numpy as np

In [25]:
df = pd.read_csv('hr_dataset.csv')

In [26]:
# quisi-identifiers to generalize and sensitive attribute
QI = ['age', 'zip_code', 'gender', 'department']
SENSITIVE = ['salary']
K = 5 

In [27]:
def generalize_age(age, range_size=10):
    """ Generalize age into ranges: 20-29, 30-39 """
    lower = (age // range_size) * range_size
    return f"{lower}-{lower + range_size -1}"

In [28]:
def generalize_zip(zip_code, digits=3):
    """ Truncate zip to first n digits: 94065 to 940**"""
    return str(zip_code)[:digits] + '*' * (5 - digits)

In [29]:
def anonymize(df, k, age_range=10, zip_digits=3):
    df = df.copy()
    df['age'] = df['age'].apply(lambda x: generalize_age(x, age_range))
    df['zip_code'] = df['zip_code'].apply(lambda x: generalize_zip(x, zip_digits))
    return df

In [33]:
def check_k_anonymity(df, qi_cols, k):
    """ Check if dataset satisfies k-anonymity"""
    groups = df.groupby(qi_cols).size().reset_index(name='count')
    violations = groups[groups['count'] < k]
    satisfies = len(violations) == 0
    print(f"K={k} anonymity satisfied: {satisfies}")
    print(f"Total groups: {len(groups)}")
    print(f"Violations (groups < k): {len(violations)}")
    print(f"Smallest group size: {groups['count'].min()}")
    return satisfies, violations

In [31]:
# start with k=5
anon_df = anonymize(df, k=K)
satisfies, violations = check_k_anonymity(anon_df, QI, K)

K=5 anonymity satisfied: False
Total groups: 4776
Violations (groups < k): 4776
Smallest group size: 1


# Observation
- K=5 anonymity satisfied: False
- Total groups: 4776
- Violations (groups < k): 4776
- Smallest group size: 1
- Given the zip codes are nearly unique (4873/5000), truncating to 3 digits is not enough to satisfy k=5, we will need to try 2 digits or even dropping zip code entirely.

In [34]:
# zip_digits = 2
def anonymize(df, k, age_range=10, zip_digits=2):
    df = df.copy()
    df['age'] = df['age'].apply(lambda x: generalize_age(x, age_range))
    df['zip_code'] = df['zip_code'].apply(lambda x: generalize_zip(x, zip_digits))
    return df

In [35]:
# try zip_digit = 2
anon_df = anonymize(df, k=K)
satisfies, violations = check_k_anonymity(anon_df, QI, K)

K=5 anonymity satisfied: False
Total groups: 3321
Violations (groups < k): 3304
Smallest group size: 1


# Observation
- zip_digit = 2 is still not enough
- Try age_range=5 compared to age_range=10

In [38]:
# age_range=5
def anonymize(df, k, age_range=5, zip_digits=2):
    df = df.copy()
    df['age'] = df['age'].apply(lambda x: generalize_age(x, age_range))
    df['zip_code'] = df['zip_code'].apply(lambda x: generalize_zip(x, zip_digits))
    return df

In [39]:
# age_range=5
anon_df = anonymize(df, k=K)
satisfies, violations = check_k_anonymity(anon_df, QI, K)

K=5 anonymity satisfied: False
Total groups: 4024
Violations (groups < k): 4023
Smallest group size: 1


# Observations
- With smaller age range, 5 compared to 10, the violations increased, k_anonymity satisfaction decreased.
- While changing zip_digit to 2 from 3 lowered the violations and increased the k_anonymity satisfaction.
- For minimum information loss, I'd remove zip_code from the quasi-identifiers list and keep the age_range high.

In [43]:
# Test 1: Dropping zip_code
QI_NO_ZIP = ['age', 'gender', 'department']

def anonymize_no_zip(df, age_range=10):
    df = df.copy()
    df['age'] = df['age'].apply(lambda x: generalize_age(x, age_range))
    return df

anon_no_zip = anonymize_no_zip(df, age_range=10)
satisfies, violations = check_k_anonymity(anon_no_zip, QI_NO_ZIP, K)

K=5 anonymity satisfied: False
Total groups: 90
Violations (groups < k): 11
Smallest group size: 1


# Observation
- Dropping the zip_code lowered the unique groups to 90 (from 5000)
- We still have 11 viloations at k=5 (we have engineers 60-69 yrs old and this is a privacy risk, even without zip code) 

In [44]:
# Test2: k=10
satisfies_k10, violations_k10 = check_k_anonymity(anon_no_zip, QI_NO_ZIP, 10)

K=10 anonymity satisfied: False
Total groups: 90
Violations (groups < k): 18
Smallest group size: 1


In [45]:
# Test 3: Measure information loss for both
print(f"\nWith zip (3 digits): {df[['age','zip_code','gender','department']].drop_duplicates().shape[0]} unique groups")
print(f"Without zip: {anon_no_zip[QI_NO_ZIP].drop_duplicates().shape[0]} unique groups")


With zip (3 digits): 5000 unique groups
Without zip: 90 unique groups


In [46]:
# subpress the violations by removing records belong to groups smaller than k
def suppress_violations(df, qi_cols, k):
    """Remove records that belong to groups smaller than k"""
    group_sizes = df.groupby(qi_cols)[qi_cols[0]].transform('count')
    suppressed = df[group_sizes >= k].copy()
    suppressed_count = len(df) - len(suppressed)
    print(f"Records suppressed: {suppressed_count} ({suppressed_count/len(df):.1%} of dataset)")
    return suppressed

# Apply suppression to fix remaining violations
anon_suppressed = suppress_violations(anon_no_zip, QI_NO_ZIP, K)
satisfies_final, _ = check_k_anonymity(anon_suppressed, QI_NO_ZIP, K)

# Final information loss measurement
print(f"\nOriginal records: {len(df)}")
print(f"After anonymization + suppression: {len(anon_suppressed)}")
print(f"Records retained: {len(anon_suppressed)/len(df):.1%}")
print(f"\nOriginal unique QI groups: {df[['age','gender','department']].drop_duplicates().shape[0]}")
print(f"Anonymized unique QI groups: {anon_suppressed[QI_NO_ZIP].drop_duplicates().shape[0]}")


Records suppressed: 26 (0.5% of dataset)
K=5 anonymity satisfied: True
Total groups: 79
Violations (groups < k): 0
Smallest group size: 5

Original records: 5000
After anonymization + suppression: 4974
Records retained: 99.5%

Original unique QI groups: 703
Anonymized unique QI groups: 79


# Observations
- Even after removing zip_code and generalizing age into ranges, 11 groups still violated k=5,primarily rare combinations of minority gender values with specific department/age combinations.
- Suppression removes these records entirely, with 100% privacy guarantee for remaining records. The tradeoff: Potential bias introduced if suppressed records are systematically different from retained ones, for example: if Non-binary employees are disproportionately suppressed, the anonymized dataset underrepresents them.

In [48]:
# check who got suppressed
suppressed_records = anon_no_zip[
    anon_no_zip.index.isin(
        set(anon_no_zip.index) - set(anon_suppressed.index)
    )
]

print("Suppressed records by gender:")
print(suppressed_records['gender'].value_counts())
print(f"\nOriginal gender distribution:")
print(df['gender'].value_counts(normalize=True).round(3))
print(f"\nRetained gender distribution:")
print(anon_suppressed['gender'].value_counts(normalize=True).round(3))

Suppressed records by gender:
gender
M             11
F              8
Non-binary     7
Name: count, dtype: int64

Original gender distribution:
gender
Non-binary    0.336
F             0.335
M             0.329
Name: proportion, dtype: float64

Retained gender distribution:
gender
Non-binary    0.337
F             0.335
M             0.329
Name: proportion, dtype: float64
